In [ ]:
# ROBUST OPTIMIZATION PROJECT

In [ ]:
# Created on 11.03.2024 by Davide Carecci

In [2]:
import sys
print(sys.path)

['C:\\Users\\lenovo\\OneDrive - Politecnico di Milano\\Work_cloud\\DOTTORATO\\Robust Optimization\\Project', 'C:\\ModelonImpact-1.8.1\\oct-dist\\install\\Python', 'C:\\Users\\lenovo\\OneDrive - Politecnico di Milano\\Work_cloud\\DOTTORATO\\Robust Optimization\\Project', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\python37.zip', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\DLLs', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\lib', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv', '', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\win32', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\win32\\lib', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\Pythonwin', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\IPython\\extensions', 'C:\\Users\\lenovo\\.ipython']


In [3]:
# Add a new path to sys.path
new_path = 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\Lib\\site-packages'
sys.path.append(new_path)

In [6]:
import rsome as rso
import numpy as np
import pandas as pd
from rsome import ro
from rsome import msk_solver as my_solver  #Import Mosek solver interface
import matplotlib.pyplot as plt
#from rsome import grb_solver as my_solver  #Import Gurobi solver interface

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Define the differential equations
def model(state, t, k1, k2):
    x, y, z = state
    dxdt = -k1 * x
    dydt = -k2 * y
    dzdt = k1 * x + k2 * y
    return [dxdt, dydt, dzdt]

def evaluate_model(x0,y0,k1,k2):

    # Initial conditions
    x0 = x0
    y0 = y0
    z0 = 0.0

    # Time points to solve the equations
    t = np.linspace(0, 30, 30)

    # Rate constants
    k1 = k1
    k2 = k2

    # Solve the differential equations
    state = odeint(model, [x0, y0, z0], t, args=(k1, k2))
    x_response = state[:, 0]
    y_response = state[:, 1]
    z_response = state[:, 2]
    
    return t,z_response

In [68]:
# LOAD DATA
# Few data are available to estimate distribution...hypothesis? Theorem to be conservative
# Load data if they already exist for further manipulation
def read_excel_file(file_path):
    # Read the Excel file and return a dictionary of DataFrames for each sheet
    xls = pd.ExcelFile(file_path)
    sheet_dict = {sheet_name: pd.read_excel(xls, sheet_name, skiprows=[1]) for sheet_name in xls.sheet_names}
    return sheet_dict

# Replace 'your_file.xlsx' with the actual file path
file_path = 'C:/Users/lenovo/OneDrive - Politecnico di Milano/Work_cloud/DOTTORATO/Robust Optimization/Project/Project_database - Copia.xlsx'

# Step 1: Read Excel file and create a dictionary of DataFrames for each sheet
data = read_excel_file(file_path)

In [69]:
# Declare constants and parameters
n = 3 #number of items (co-substrates)
m = 5 #number of suppliers

Vr = 6390 #m3 reactor volume----to be modified. SAntonio?
N = 90 #purchase horizon (days)
p_ch4 = 110 #fixed biomethane price
p_ch4 = 110 *0.01035 #Conversion MWh/Nm3CH4

bnds = np.array([[0, 120], #VS
        [0.2, 4], #TKN
        [20, 40], #C/N
        [0, 4], #TAN
        [5, 15], #TAC
        [0, 10], #LI
        [0, 10], #TVFA
        [20,40] #HRT
       ])

In [74]:
k_hydr = data['Item characteristics'].values[:,13].astype('float64')
k_hyds = data['Item characteristics'].values[:,14].astype('float64')
BMPinf_r = data['Item characteristics'].values[:,15].astype('float64')
BMPinf_s = data['Item characteristics'].values[:,16].astype('float64')

In [ ]:
# SAMPLE BMP CURVE 
bmp_curve = []
for k1,k2,x0,y0 in zip(k_hydr,k_hyds,BMPinf_r,BMPinf_s):
    t,z_response = evaluate_model(x0,y0,k1,k2)
    bmp_curve.append(z_response)

    # Calculate dz/dt using numerical differentiation
    dt = t[1] - t[0]
    dzdt_response = np.gradient(z_response, dt)

    # Plot the time response
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(t, x_response, label='x(t)')
    ax.plot(t, y_response, label='y(t)')
    ax.plot(t, z_response, label='z(t)', marker= 'o')
    ax.plot(t, dzdt_response, label='dz/dt')
    plt.xlabel('Time')
    plt.ylabel('Concentration / Rate')
    plt.title('Time Response of State Variables and their Rates')
    plt.legend()
    plt.grid(True)
    #plt.show()
bmp_curve = np.array(bmp_curve, dtype=np.float64)
#print(bmp_curve)

In [75]:
# Re-declaration of loaded data
A = data['Availability'].values[:,1:].astype('float64')
k = np.concatenate(([data['Item characteristics'].values[:,3]], data['Item characteristics'].values[:,6:12].T), axis=0).T
k = k.astype('float64')

VS = data['Item characteristics']['VS'].values.astype('float64')
BMP = data['Item characteristics']['BMPinf'].values.astype('float64') #We must sample this from the curve fixing HRT
Cp = data['Purchase prices'].values[:,1:].astype('float64')
Ct = data['Item characteristics']['Ct'].values.astype('float64')
d = data['Distance']['d'].values.astype('float64')

In [152]:
# Just for checking purposes
f_obj = p_ch4*(VS*BMP/1000)@x.sum(axis=1)
f_obj2 = (Cp.T*x).sum()
f_obj3 = Ct@(d@x.T)

In [153]:
# Just for checking purposes
#print(k[:,1])
for i in range(np.size(k,1)):
    ciao = k[:,i]@(x.sum(axis=1))
    print(ciao)

3450599.9999999995
93538.95
764244.0
35677.5
213000.0
139781.25
181050.0


In [154]:
#Solve the nominal problem

#Create model
model=ro.Model('Biomethane supply-chain')

#Define variables
x = model.dvar((n,m))          #

#List the objective and constraints
model.max(p_ch4*(VS*BMP/1000)@x.sum(axis=1)-(Cp.T*x).sum()-Ct@(d@x.T))
#model.st(x <= A)       # Stock constraint
model.st(x <= A.T)       # Availability constraint
for i in range(np.size(k,1)):
    model.st(k[:,i]@(x.sum(axis=1)) <= bnds[i][1]*x.sum()) # Technical constraint (max)
    model.st(k[:,i]@(x.sum(axis=1)) >= bnds[i][0]*x.sum()) # Technical constraint (min)
model.st(Vr*N <= bnds[7][1]*x.sum()) # HRT constraint (max)
model.st(Vr*N >= bnds[7][0]*x.sum()) # HRT constraint (min)
model.st(x >= 0)

#Solve the model
model.solve(my_solver)
opt_obj = model.get()  #
opt_x = x.get()

print('The objective is', opt_obj, 'and the optimal faciliy location is', opt_x)

Being solved by Mosek...
Solution status: optimal
Running time: 0.0963s
The objective is 628462.46 and the optimal faciliy location is [[    0.     0.     0.     0.     0.]
 [16300.     0.  5000.     0.     0.]
 [    0.     0.  7455.     0.     0.]]


In [145]:
# Just for checking purposes
x = opt_x
f_obj = p_ch4*(VS*BMP/1000)@x.sum(axis=1)
f_obj2 = (Cp.T*x).sum()
f_obj3 = Ct@(d@x.T)
print(f_obj)
print(f_obj2)
print(f_obj3)
print(f_obj-f_obj2-f_obj3)

1193102.46
448274.99999999994
116364.99999999999
628462.46


In [ ]:
# Design of uncertainty set
# Budgeted set
# Normal distribution theorem to compute Gamma? No
# If data, have to scale them between [-1,1]? Yes